# Exploratory analysis: IEEE-CIS fraud data

This notebook is the **exploration** behind the project: imbalance, time structure, missing data and amount patterns.
The modelling, rules baseline, cost evaluation and all reported numbers live in the reproducible pipeline (`python -m fraud.pipeline`), not here, so the README figures always come from one place.

**Data:** download `train_transaction.csv` and `train_identity.csv` from [Kaggle](https://www.kaggle.com/c/ieee-fraud-detection/data) into `data/`.


In [ ]:
import os
import sys

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd
import seaborn as sns

sys.path.insert(0, os.path.abspath("../src"))
from fraud.data import load_raw

sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
pd.set_option("display.max_columns", 50)
SEED = 42


---

## Phase 1: Data Loading

The IEEE-CIS dataset comes in two tables that need to be joined:

1. **train_transaction.csv** — every transaction, with amount, card info, time, and 339 `V` features
2. **train_identity.csv** — device/browser/email signals for a *subset* of transactions (not every transaction has identity data — about 144k of 590k do)

We join them on `TransactionID` using a **left join** — keeping all transactions and filling identity gaps with NaN.

> In production, identity signals often arrive asynchronously — a transaction fires, and the device fingerprint comes in milliseconds later. A left join mirrors this: you score the transaction with whatever you have, and identity features gracefully degrade to NaN when absent. Dropping those rows (inner join) would silently bias your training data.


In [ ]:
DATA_DIR = os.getenv("FRAUD_DATA_DIR", "../data")
df = load_raw(DATA_DIR, "train_transaction.csv", "train_identity.csv")
print(f"Merged dataset: {df.shape[0]:,} rows x {df.shape[1]} columns")


---

## Phase 2: Exploratory Data Analysis

EDA is not optional decoration. It's how you catch problems before they silently destroy model performance.

We'll look at:
1. **Class imbalance** — how rare is fraud?
2. **Temporal structure** — do transactions have a time axis we need to respect?
3. **Missing data** — which features are usable and which are mostly empty?
4. **Transaction amount distribution** — does fraud cluster at certain values?



In [ ]:
# ── 2.1  Target distribution — how imbalanced is this problem? ────────────────

fraud_counts = df['isFraud'].value_counts()
fraud_rate   = df['isFraud'].mean() * 100

print("=" * 45)
print("  CLASS DISTRIBUTION")
print("=" * 45)
print(f"  Legitimate (0) : {fraud_counts[0]:>8,}  ({100 - fraud_rate:.2f}%)")
print(f"  Fraud      (1) : {fraud_counts[1]:>8,}  ({fraud_rate:.2f}%)")
print("=" * 45)
print(f"\n  Imbalance ratio : ~{fraud_counts[0]//fraud_counts[1]}:1  (legit:fraud)")

# ── Why imbalance matters ─────────────────────────────────────────────────────
# A model that predicts EVERY transaction as legitimate would score ~96.5%
# accuracy. That's useless for fraud detection. This is why accuracy is a
# misleading metric here — we'll use Precision-Recall AUC instead.

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Bar chart of raw counts
axes[0].bar(['Legitimate', 'Fraud'], fraud_counts.values,
             color=['#4C72B0', '#DD8452'], edgecolor='white', linewidth=1.5)
axes[0].set_title('Transaction Counts by Class', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Number of Transactions')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))

# Pie chart — easier to show the proportion visually
axes[1].pie(fraud_counts.values,
             labels=['Legitimate', 'Fraud'],
             autopct='%1.2f%%',
             colors=['#4C72B0', '#DD8452'],
             startangle=90,
             wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[1].set_title('Class Proportion', fontsize=13, fontweight='bold')

plt.suptitle('Target Variable: isFraud', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ── 2.2  Temporal structure — the most important EDA step for production ───────

# TransactionDT is seconds elapsed since some reference point (not a real
# timestamp, but it preserves the ordering of transactions in time)

print("TransactionDT range:")
print(f"  Min : {df['TransactionDT'].min():,} seconds")
print(f"  Max : {df['TransactionDT'].max():,} seconds")
print(f"  Span: {(df['TransactionDT'].max() - df['TransactionDT'].min()) / 86400:.1f} days")

# Convert to approximate day number for plotting
df['day'] = df['TransactionDT'] // 86400  # 86400 seconds = 1 day

# Count transactions and fraud per day
daily = df.groupby('day').agg(
    total_txns   = ('isFraud', 'count'),
    fraud_txns   = ('isFraud', 'sum')
).reset_index()
daily['fraud_rate'] = daily['fraud_txns'] / daily['total_txns']

fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

# Top panel: daily transaction volume
axes[0].plot(daily['day'], daily['total_txns'], color='#4C72B0', linewidth=1.5)
axes[0].fill_between(daily['day'], daily['total_txns'], alpha=0.2, color='#4C72B0')
axes[0].set_ylabel('Transactions per Day')
axes[0].set_title('Daily Transaction Volume', fontweight='bold')

# Bottom panel: daily fraud rate — look for spikes or drift
axes[1].plot(daily['day'], daily['fraud_rate'] * 100, color='#DD8452', linewidth=1.5)
axes[1].fill_between(daily['day'], daily['fraud_rate'] * 100, alpha=0.2, color='#DD8452')
axes[1].set_ylabel('Fraud Rate (%)')
axes[1].set_xlabel('Day (relative to dataset start)')
axes[1].set_title('Daily Fraud Rate', fontweight='bold')

plt.suptitle('Temporal Structure of the Dataset', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("\n⚠️  Key insight: This dataset spans ~6 months.")
print("   We MUST split train/test by time — not randomly.")
print("   A random split would leak future information into training.")
print("   In production, your model never sees the future.")

In [ ]:
# ── 2.3  Missing data audit — which features are usable? ─────────────────────

# Calculate % missing per column
missing = (df.isnull().sum() / len(df) * 100).sort_values(ascending=False)

# Split into categories for readability
high_missing   = missing[missing > 70]   # essentially unusable without imputation
medium_missing = missing[(missing > 20) & (missing <= 70)]
low_missing    = missing[(missing > 0)   & (missing <= 20)]
complete       = missing[missing == 0]

print(f"Columns with >70%  missing : {len(high_missing):>4}  (we'll drop or flag these)")
print(f"Columns with 20–70% missing: {len(medium_missing):>4}  (impute with median/mode)")
print(f"Columns with <20%  missing : {len(low_missing):>4}  (impute or keep as-is)")
print(f"Complete columns           : {len(complete):>4}")

# Visualise the top 40 worst columns
fig, ax = plt.subplots(figsize=(14, 8))
top_missing = missing[missing > 0].head(40)

colors = ['#c0392b' if v > 70 else '#e67e22' if v > 20 else '#f1c40f'
          for v in top_missing.values]

ax.barh(top_missing.index[::-1], top_missing.values[::-1], color=colors[::-1])
ax.axvline(70, color='#c0392b', linestyle='--', alpha=0.7, label='>70% (drop)')
ax.axvline(20, color='#e67e22', linestyle='--', alpha=0.7, label='>20% (careful imputation)')
ax.set_xlabel('% Missing')
ax.set_title('Top 40 Columns by Missing Data %', fontsize=13, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ── 2.4  Transaction amount — does fraud look different? ──────────────────────

# Log scale because transaction amounts are extremely right-skewed
# (a few huge transactions, many small ones)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: distribution by class on log scale
for fraud_label, color, label in [(0, '#4C72B0', 'Legitimate'), (1, '#DD8452', 'Fraud')]:
    subset = df[df['isFraud'] == fraud_label]['TransactionAmt']
    axes[0].hist(np.log1p(subset), bins=60, alpha=0.6, color=color, label=label)

axes[0].set_xlabel('log(1 + TransactionAmt)')
axes[0].set_ylabel('Count')
axes[0].set_title('Transaction Amount Distribution\n(log scale)', fontweight='bold')
axes[0].legend()

# Right: box plot for cleaner comparison of spread
plot_data = df[['TransactionAmt', 'isFraud']].copy()
plot_data['Class'] = plot_data['isFraud'].map({0: 'Legitimate', 1: 'Fraud'})
plot_data['log_amt'] = np.log1p(plot_data['TransactionAmt'])

sns.boxplot(data=plot_data, x='Class', y='log_amt', ax=axes[1],
            palette={'Legitimate': '#4C72B0', 'Fraud': '#DD8452'})
axes[1].set_ylabel('log(1 + TransactionAmt)')
axes[1].set_title('Amount Distribution by Class', fontweight='bold')

plt.suptitle('Do Fraudulent Transactions Have Different Amounts?', 
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# Summary stats
print("\nMedian transaction amount:")
print(df.groupby('isFraud')['TransactionAmt'].median().rename({0: 'Legitimate', 1: 'Fraud'}))

## What this means for the modelling

- **Time matters**: fraud rate and volume move over the six months, so the pipeline splits train → validation → test strictly by time.
- **Heavy missingness**: columns more than 80% empty (measured on the training period only) are dropped; missingness of identity fields is kept as a signal.
- **Amounts differ by class**: so a missed fraud is priced at its actual amount, not a flat average.

Next: `python -m fraud.pipeline` (see README).
